## Image preparation

The below code can be used to transform the images in an input directory (Input_Dir) 
to the right size (e.g. 32x32 pixels) into an output directory (Output_Dir).

Note: Duplicates will be removed by evaluating the file hash

### Basic Parameter

IMPORTANT: Do not rename any variables in this section — they are externally referenced in the GitHub action `Train Model`.

* `Input_Dir`: Path to the input directory containing training images
* `Output_Dir`: Path to the output directory where results (models, logs, etc.) will be saved
* `Target_Shape`: Tuple specifying the image dimensions (width, height, channels)

In [1]:
# Parameters
Input_Dir = 'data_raw_all'
Output_Dir = 'data_resize_all'

# Target image size [x, y, channels]
Target_Shape = (32, 32, 3)


In [2]:
# Parameters
Input_Dir = "data_raw_all"
Output_Dir = "data_resize_all"


### Load libraries and defaults

In [3]:
import glob
import os

from pathlib import Path
import hashlib
import tensorflow as tf


2025-05-26 19:16:47.561705: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-26 19:16:47.564699: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-26 19:16:47.571062: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748287007.585045    2184 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748287007.589247    2184 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-26 19:16:47.604489: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

### Delete output directory

In [4]:
files = glob.glob(Output_Dir + '/*.jpg')
for f in files:
    os.remove(f)
print(str(len(files)) + " files have been deleted.")


0 files have been deleted.


### Load files and resize

In [5]:
files = glob.glob(Input_Dir + '/*.jpg')
hashes = {}

for i, aktfile in enumerate(files):
    if i % 500 == 0:
        print(i, aktfile)

    # Read
    image_bytes = tf.io.read_file(aktfile)
    image = tf.image.decode_image(image_bytes, channels=Target_Shape[2], expand_animations=False)

    # Compute SHA256 hash of image content only, not of file itself (exclude metadata, etc)
    hash_val = hashlib.sha256(tf.io.encode_jpeg(image).numpy()).hexdigest()
    if hash_val in hashes:
        hashes[hash_val].append(aktfile)
        continue
    else:
        hashes[hash_val] = [aktfile]
    
    # Resize
    image = tf.image.resize(image, [Target_Shape[0], Target_Shape[1]], method=tf.image.ResizeMethod.MITCHELLCUBIC)
    image = tf.cast(image, tf.uint8)
    image = tf.io.encode_jpeg(image, quality=100)

    # Save
    base = os.path.basename(aktfile)
    save_path = os.path.join(Output_Dir, base)
    tf.io.write_file(save_path, image)



0 data_raw_all/6.1_a1500e3f3187c4cede044a4487e02b7d.jpg


2025-05-26 19:16:51.363074: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


500 data_raw_all/7.0_1e781872e114eb9bb865763fa19dd893.jpg


1000 data_raw_all/3.2_1537_zeiger2_2019-06-01T174233.jpg


1500 data_raw_all/8.5_3b3abd4281ab6190f8d3d8e8bcdf7064.jpg


### Remove duplicate files

In [6]:
# duplicate files are a risk to the metrics, they pollute the validation dataset
for hash in hashes:
    if len(hashes[hash]) > 1:
        print(f"Duplicates: {hashes[hash]}")    
        for duplicate in hashes[hash][1:]:
            # remove all except the first
            print(f"Removed: {duplicate}")
            os.remove(duplicate)    
